In [ ]:
from brian2 import *

import numpy as np
import os
from glob import glob
import cv2
import sys
sys.path.insert(0, r"C:\Users\reidj\Dropbox\dphil\programming\spikes\spikes")
from network import *
from input import *
from projects import *


# Section 1: Define Functions we'll need
## Section 1.1: Input preparation functions

In [ ]:
def load_gabor_filters(filter_dir):
    """
    Load all Gabor filter .npy files from a directory into a list of arrays.
    
    Args:
        filter_dir (str): Path to directory containing filter .npy files
        
    Returns:
        list: List of numpy arrays, each representing a Gabor filter
    """
    # Ensure path exists
    if not os.path.exists(filter_dir):
        raise FileNotFoundError(f"Filter directory not found: {filter_dir}")
    
    # Get all .npy files in the directory
    filter_files = glob(os.path.join(filter_dir, "*.npy"))
    
    if not filter_files:
        print(f"No .npy files found in {filter_dir}")
        return []
    
    # Load each filter into a list
    filters = []
    for file_path in filter_files:
        try:
            filter_array = np.load(file_path)
            filters.append(filter_array)
            print(f"Loaded filter from {os.path.basename(file_path)}, shape: {filter_array.shape}")
        except Exception as e:
            print(f"Error loading {file_path}: {str(e)}")
    
    print(f"Loaded {len(filters)} Gabor filters")
    return filters

def upscale_mnist(images, target_size=128, method='bicubic'):
    """
    Upscale MNIST images from 28x28 to target_size x target_size
    
    Args:
        images: NumPy array with shape (n_images, 28, 28)
        target_size: Target size (default: 128)
        method: Upscaling method ('nearest', 'bilinear', 'bicubic', or 'lanczos')
        
    Returns:
        NumPy array with shape (n_images, target_size, target_size)
    """
    num_images = images.shape[0]
    upscaled = np.zeros((num_images, target_size, target_size))
    
    for i in range(num_images):
        # OpenCV resize
        if method == 'nearest':
            interpolation = cv2.INTER_NEAREST
        elif method == 'bilinear':
            interpolation = cv2.INTER_LINEAR
        elif method == 'bicubic':
            interpolation = cv2.INTER_CUBIC
        elif method == 'lanczos':
            interpolation = cv2.INTER_LANCZOS4
        else:
            raise ValueError(f"Unknown method: {method}")
        
        # OpenCV takes (width, height) instead of (height, width)
        upscaled[i] = cv2.resize(images[i], (target_size, target_size), interpolation=interpolation)
        
    return upscaled


## 1.2: Visualisation of Input:


In [ ]:
def display_convolved_images(convolved_data, cmap='viridis', figsize=(15, 10)):
    """
    Display convolved images as heatmaps in a grid layout.
    
    Args:
        convolved_data: NumPy array with shape (n_images, n_filters, height, width)
        cmap: Colormap for the heatmap (default: 'viridis')
        figsize: Figure size (width, height) in inches
    """
    n_images, n_filters, height, width = convolved_data.shape
    
    # Create a figure with subplots - one row per image, one column per filter
    fig, axes = plt.subplots(n_images, n_filters, figsize=figsize)
    
    # Find global min and max for consistent color scaling across all heatmaps
    vmin = np.min(convolved_data)
    vmax = np.max(convolved_data)
    
    # Plotting each image × filter combination
    for img_idx in range(n_images):
        for filter_idx in range(n_filters):
            # Get current axis
            ax = axes[img_idx, filter_idx]
            
            # Display the heatmap
            im = ax.imshow(convolved_data[img_idx, filter_idx], 
                          cmap=cmap, 
                          vmin=vmin, vmax=vmax)
            
            # Remove axis ticks for cleaner display
            ax.set_xticks([])
            ax.set_yticks([])
            
            # Add labels for first row and column only
            if img_idx == 0:
                ax.set_title(f"Filter {filter_idx}")
            if filter_idx == 0:
                ax.set_ylabel(f"Image {img_idx}")
    
    # Add a colorbar
    cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6)
    cbar.set_label('Activation Value')
    
    # Add overall title
    plt.suptitle("Convolved Images (MNIST) with Gabor Filters", fontsize=16)
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)  # Make room for suptitle
    
    return fig

def display_neuron_inputs(neuron_inputs, cmap='hot', figsize=(12, 8)):
    """
    Display neuron inputs as heatmaps.
    
    Args:
        neuron_inputs: NumPy array with shape (n_images, neuron_count) or (n_images, height, width)
        cmap: Colormap for the heatmap (default: 'hot')
        figsize: Figure size (width, height) in inches
    """
    # Check shape and reshape if needed
    if len(neuron_inputs.shape) == 2:
        # We need to estimate the grid size
        n_images, neuron_count = neuron_inputs.shape
        grid_size = int(np.sqrt(neuron_count))
        
        if grid_size**2 != neuron_count:
            print(f"Warning: Neuron count {neuron_count} is not a perfect square.")
            grid_size = int(np.sqrt(neuron_count))
            # Pad with zeros to make it square
            padded_count = grid_size**2
            padded_inputs = np.zeros((n_images, padded_count))
            padded_inputs[:, :neuron_count] = neuron_inputs
            neuron_inputs = padded_inputs.reshape(n_images, grid_size, grid_size)
        else:
            # Reshape to grid
            neuron_inputs = neuron_inputs.reshape(n_images, grid_size, grid_size)
    
    n_images = neuron_inputs.shape[0]
    
    # Calculate grid layout for subplots
    n_cols = min(5, n_images)
    n_rows = (n_images + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_rows == 1 and n_cols == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    # Find global min and max for consistent color scaling
    vmin = np.min(neuron_inputs)
    vmax = np.max(neuron_inputs)
    
    # Plot each image
    for i in range(n_images):
        ax = axes[i]
        im = ax.imshow(neuron_inputs[i], cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(f"Image {i}")
        ax.set_xticks([])
        ax.set_yticks([])
    
    # Hide unused subplots
    for i in range(n_images, len(axes)):
        axes[i].axis('off')
    
    # Add colorbar
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label('Neuron Input Strength')
    
    plt.suptitle("Neuron Input Activations", fontsize=16)
    plt.tight_layout(rect=[0, 0, 0.9, 0.95])
    
    return fig

# 2: Load in Data:
## 2.1: Load in MNIST

In [ ]:

import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
# 1) Load the train split as a tf.data.Dataset of (image, label) pairs
ds_train = tfds.load(
    "mnist",
    split="train",
    as_supervised=True,    # yields (image, label) tuples
    shuffle_files=False,   # no need to shuffle if you just want the first 10
)

# 2) Take only the first 10 examples
ds20 = ds_train.take(20)

# 3) Convert to NumPy and stack
#    - If you’re in Eager mode (TF2 default), .numpy() works on each tensor
mnist_images = np.stack([img.numpy() for img, lbl in ds20])
# mnist_images.shape == (120, 28, 28, 1)

# 4) (Optional) squeeze off the channel dimension if you want (10,28,28)
mnist_images = mnist_images.squeeze(-1)

# 5) Upscale:
upscaled_images = upscale_mnist(mnist_images, target_size=128, method='bicubic')
print(f"Original shape: {mnist_images.shape}, Upscaled shape: {upscaled_images.shape}")



## 2.2. Load in Gabor filters & convolve images, saving them to a file location:

In [ ]:
filter_dir = r"C:\Users\reidj\Dropbox\dphil\programming\spikes\projects\mnist_class\mnist_class_wip\configs\input\filters"
gabor_filters = load_gabor_filters(filter_dir)
convolved_images = convolve_images(upscaled_images, 
                    gabor_filters, 
                    r"C:\Users\reidj\Dropbox\dphil\programming\spikes\projects\mnist_class\mnist_class_wip\data\conv_mnist_20"
                    )


## 2.3: Now Generate Neuron Inputs by Combining the Inputs with the mapping:


In [ ]:
neuron_inputs = generate_neuron_inputs_from_saved(r"C:\Users\reidj\Dropbox\dphil\programming\spikes\projects\mnist_class\mnist_class_wip\data\conv_mnist_20.npy",
                                    r"C:\Users\reidj\Dropbox\dphil\programming\spikes\projects\mnist_class\mnist_class_wip\configs\input\mapping.npz")

In [ ]:
print(neuron_inputs.shape)

# 3: Now create the Network following the config already prespecified in the config/epoch_0 file
## 3.1: Set up parameters

In [ ]:
set_device('cpp_standalone', build_on_run=False)
equations_container = EquationsContainer()
network = Network()
project = "mnist_class_wip"
dir = r"C:\Users\reidj\Dropbox\dphil\programming\spikes\projects\mnist_class\mnist_class_wip\code\scripts"
config_dir = os.path.join(dir, "..\..\configs")
# Set up parameters
N_layers = 4
STIMULUS_LENGTH = 100 * ms
image_height, image_width, num_filters = 128, 128, 8
grid_size = 64  # For excitatory layers
RADII = {
    "efe": {1: 8, 2: 12, 3: 16},
    "ele": {1: 2, 2: 2, 3: 2, 4: 2},
    "ebe": {2: 8, 3: 8, 4: 8},
    "eli": {1: 2, 2: 2, 3: 2, 4: 2},
    "ile": {1: 4, 2: 4, 3: 4, 4: 4},
}

AVG_NO_CONNECTIONS = {
    "efe": {0: 50, 1: 100, 2: 100, 3: 100},
    "ele": {1: 10, 2: 10, 3: 10, 4: 10},
    "ebe": {1: 10, 2: 10, 3: 10, 4: 10},
    "eli": {1: 10, 2: 10, 3: 10, 4: 10},
    "ile": {1: 30, 2: 30, 3: 30, 4: 30},
}

print("Defining Excitatory Neurons")
exc_neuron_specs = NeuronSpecs(
    neuron_type="e",
    length=64,
    cm=500 * pF,
    g_leak=25 * nS,
    v_threshold=-53 * mV,
    v_reset=-57 * mV,
    v_rest=-74 * mV,
    v_reversal_e=0 * mV,
    v_reversal_i=-70 * mV,
    v_reversal_a=-90 * mV,
    sigma=0.015 * mV,
    t_refract=2 * ms,
    tau_m=20 * ms,
    tau_ee=2 * ms,
    tau_ie=5 * ms,
    tau_a=80 * ms,
)
print("Defining Inhibitory Neurons")
inh_neuron_specs = NeuronSpecs(
    neuron_type="i",
    length=32,
    cm=214 * pF,
    g_leak=18 * nS,
    v_threshold=-53 * mV,
    v_reset=-58 * mV,
    v_rest=-82 * mV,
    v_reversal_e=0 * mV,
    v_reversal_i=-70 * mV,
    sigma=0.015 * mV,
    t_refract=2 * ms,
    tau_m=12 * ms,
    tau_ei=2 * ms,
    tau_ii=5 * ms,
)
print("Defining Synapse Specifications")
print("Defining EFE")
efe_synapse_specs = SynapseSpecs(
    model=equations_container.synaptic_equations["stdp_model"],
    on_pre=equations_container.synaptic_equations["stdp_on_pre"],
    on_post=equations_container.synaptic_equations["stdp_on_post"],
    type="f",
    name="efe",
    lambda_e=30 * nS,
    alpha_C=0.5,
    alpha_D=0.5,
    tau_c=5 * ms,
    tau_d=5 * ms,
    learning_rate=0.04,
)
print("Defining ELE")
ele_synapse_specs = SynapseSpecs(
    model=equations_container.synaptic_equations["stdp_model"],
    on_pre=equations_container.synaptic_equations["stdp_on_pre"],
    on_post=equations_container.synaptic_equations["stdp_on_post"],
    type="l",
    name="ele",
    lambda_e=20 * nS,
    alpha_C=0.5,
    alpha_D=0.5,
    tau_c=5 * ms,
    tau_d=5 * ms,
    learning_rate=0.04,
)
print("Defining EBE")
ebe_synapse_specs = SynapseSpecs(
    model=equations_container.synaptic_equations["stdp_model"],
    on_pre=equations_container.synaptic_equations["stdp_on_pre"],
    on_post=equations_container.synaptic_equations["stdp_on_post"],
    type="b",
    name="ebe",
    lambda_e=20 * nS,
    alpha_C=0.5,
    alpha_D=0.5,
    tau_c=5 * ms,
    tau_d=5 * ms,
    learning_rate=0.04,
)
print("Defining ELI")
eli_synapse_specs = SynapseSpecs(
    model=equations_container.synaptic_equations["excit_non_stdp_model"],
    on_pre=equations_container.synaptic_equations["excit_non_stdp_on_pre"],
    type="l",
    name="eli",
    lambda_e=20 * nS,
)
print("Defining ILE")
ile_synapse_specs = SynapseSpecs(
    model=equations_container.synaptic_equations["inhib_non_stdp_model"],
    on_pre=equations_container.synaptic_equations["inhib_non_stdp_on_pre"],
    type="l",
    name="ile",
    lambda_i=30 * nS,
)

## 3.2. Create Network

In [ ]:
print("Creating Network")
create_network(
    network,
    4,
    exc_neuron_specs,
    inh_neuron_specs,
    RADII,
    AVG_NO_CONNECTIONS,
    efe_synapse_specs,
    ele_synapse_specs,
    ebe_synapse_specs,
    eli_synapse_specs,
    ile_synapse_specs,
    storage="load",
    storage_path=os.path.join(config_dir, "network")
)
input_layer = exc_neuron_specs.neuron_groups[0]
monitor = SpikeMonitor(input_layer, bin=20 * ms)

device.build(run=False)  # Compile the code
# Load the input data

## 3.3. Run the Network

In [ ]:
no_images, width, height = neuron_inputs.shape
print("neuron_inputs shape:", neuron_inputs.shape)
for i in range(no_images):
    input = neuron_inputs[i]
    input_layer.rates = input
    network.run(0.1 * second, report="text")
    print("Running network for image", i)

